# Layers that bend boundaries
**Machine Learning · Week 5 (Neural networks with PyTorch) · Day 2**

Lesson page: https://sara-aminpour.com/tutorials/ml/w5d2.html

A **neuron** is logistic regression: a weighted sum, then a nonlinear squash. Stack layers of
neurons and the decision boundary can bend into any shape. Built and trained with PyTorch.

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons

torch.set_num_threads(1)
X, y = make_moons(n_samples=300, noise=0.2, random_state=0)
Xt = torch.tensor(X, dtype=torch.float32)
yt = torch.tensor(y, dtype=torch.float32).reshape(-1, 1)

torch.manual_seed(0)
model = torch.nn.Sequential(
    torch.nn.Linear(2, 16),   # 2 inputs  -> 16 hidden neurons
    torch.nn.Tanh(),          # the bend: a nonlinear activation
    torch.nn.Linear(16, 1),   # 16 hidden -> 1 output score
)
print(model)
print("trainable numbers:", sum(p.numel() for p in model.parameters()))

## Train it: the same five-line loop as last Friday

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.03)
loss_fn = torch.nn.BCEWithLogitsLoss()

for epoch in range(501):
    optimizer.zero_grad()
    loss = loss_fn(model(Xt), yt)
    loss.backward()
    optimizer.step()
    if epoch % 100 == 0:
        with torch.no_grad():
            accuracy = ((model(Xt) > 0).float() == yt).float().mean().item()
        print(f"epoch {epoch:3d}: loss {loss.item():.3f}, accuracy {accuracy:.3f}")

## Draw the boundary
A small helper colors the plane by the model's probability. We'll reuse it below.

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

apricot_blue = LinearSegmentedColormap.from_list("apricot_blue", ["#ff8a50", "#ffffff", "#5577f5"])

def plot_boundary(model, ax, title):
    xx, yy = np.meshgrid(np.linspace(-1.6, 2.6, 200), np.linspace(-1.1, 1.6, 200))
    grid = torch.tensor(np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32)
    with torch.no_grad():
        p = torch.sigmoid(model(grid)).reshape(xx.shape).numpy()
    ax.contourf(xx, yy, p, levels=20, cmap=apricot_blue, alpha=0.5)
    ax.contour(xx, yy, p, levels=[0.5], colors="#2c2842", linewidths=1.8)
    ax.scatter(X[:, 0], X[:, 1], c=np.where(y == 0, "#ff8a50", "#5577f5"), s=12)
    ax.set_title(title)

fig, ax = plt.subplots(layout="constrained")
plot_boundary(model, ax, "16 hidden neurons, tanh")
plt.show()

## How much network do we need?
Same data, four models: no hidden layer (that's logistic regression), then 2, 3 and 16 hidden neurons.

In [ ]:
def train(hidden, epochs=600):
    torch.manual_seed(1)
    if hidden == 0:
        net = torch.nn.Linear(2, 1)
    else:
        net = torch.nn.Sequential(torch.nn.Linear(2, hidden), torch.nn.Tanh(), torch.nn.Linear(hidden, 1))
    opt = torch.optim.Adam(net.parameters(), lr=0.03)
    for _ in range(epochs):
        opt.zero_grad()
        loss_fn(net(Xt), yt).backward()
        opt.step()
    with torch.no_grad():
        acc = ((net(Xt) > 0).float() == yt).float().mean().item()
    return net, acc

fig, axes = plt.subplots(1, 4, figsize=(13, 3.2), sharey=True, layout="constrained")
for ax, hidden in zip(axes, [0, 2, 3, 16]):
    net, acc = train(hidden)
    plot_boundary(net, ax, f"{hidden} hidden: accuracy {acc:.2f}")
plt.show()

**Try this:** replace `torch.nn.Tanh()` with `torch.nn.ReLU()` in `train` and run again: ReLU boundaries are
made of straight pieces.

## Your turn: beat XOR
Yesterday a single neuron could never learn XOR. Give it a hidden layer of just **two** neurons:
`Linear(2, 2)`, `Tanh`, `Linear(2, 1)`. Train it on the four XOR points and print its predictions.

In [ ]:
Xx = torch.tensor([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]])
yx = torch.tensor([[0.0], [1.0], [1.0], [0.0]])
torch.manual_seed(0)
xor_net = None          # TODO: nn.Sequential with a hidden layer of 2 neurons
# TODO: train with Adam (lr = 0.1) and BCEWithLogitsLoss for 1000 steps
if xor_net is not None:
    with torch.no_grad():
        print(torch.sigmoid(xor_net(Xx)).ravel().numpy().round(3))

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
torch.manual_seed(0)
xor_net = torch.nn.Sequential(torch.nn.Linear(2, 2), torch.nn.Tanh(), torch.nn.Linear(2, 1))
optimizer = torch.optim.Adam(xor_net.parameters(), lr=0.1)
loss_fn = torch.nn.BCEWithLogitsLoss()
for step in range(1000):
    optimizer.zero_grad()
    loss_fn(xor_net(Xx), yx).backward()
    optimizer.step()
with torch.no_grad():
    print("P(1) for 00, 01, 10, 11:", torch.sigmoid(xor_net(Xx)).ravel().numpy().round(3))
W, b = xor_net[0].weight.detach().numpy(), xor_net[0].bias.detach().numpy()
for k in range(2):
    print(f"hidden neuron {k + 1} draws the line {W[k, 0]:+.2f}·x1 {W[k, 1]:+.2f}·x2 {b[k]:+.2f} = 0")
print("Two lines, combined by the output neuron, cut off the two corners that XOR needs. That's the whole trick.")

# Bonus: is it always that easy? Train from 20 different random starts.
def solves_xor(hidden, seed):
    torch.manual_seed(seed)
    net = torch.nn.Sequential(torch.nn.Linear(2, hidden), torch.nn.Tanh(), torch.nn.Linear(hidden, 1))
    opt = torch.optim.Adam(net.parameters(), lr=0.1)
    for _ in range(1000):
        opt.zero_grad()
        loss_fn(net(Xx), yx).backward()
        opt.step()
    with torch.no_grad():
        return bool(((torch.sigmoid(net(Xx)) > 0.5).float() == yx).all())

for hidden in [2, 4]:
    wins = sum(solves_xor(hidden, s) for s in range(20))
    print(f"{hidden} hidden neurons: solved XOR from {wins} of 20 random starts")
print("With only 2 neurons, many starts get stuck in a local minimum; a little extra width removes most of the traps.")